# Introduction

<!-- describe general idea of what we are doing here -->


## Setting up the Virtual Environment

To install Python libraries, navigate to the folder "like_dislike" in your terminal, then run `python3 -m venv .venv` to create a new virtual environment and activate it using `source .venv/bin/activate`. Run `pip3 install -r requirements.txt` to install required Python libraries into the created virtual environment.


## Libraries

In [2]:
import pandas as pd
import os
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm.auto import tqdm

## Data-Wrangling

Next, we load the comment data.

In [5]:
# loading comments data
data_path = "data/comments/comments_reliy_coded.xlsx"
data_comments = pd.read_excel(data_path)

## Hugging Face

Hugging Face provides an open-source platform that allows data scientists, computer scientists, and researchers to share models and datasets. We will use a model from its comprehensive transformer library. To do so:

1. Create an access token on [Hugging Face](https://huggingface.co/settings/tokens).

2. Run in terminal: `echo 'export HF_TOKEN="hf_yournewtoken"' >> ~/.zshrc && source ~/.zshrc`

3. Run to verify: `echo $HF_TOKEN`

In [6]:
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Use `pipeline` "zero-shot-classification" to download the classifier model:

In [7]:
post_classifier = pipeline(
    "zero-shot-classification", 
    model="MoritzLaurer/xlm-v-base-mnli-xnli")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8780.36it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: MoritzLaurer/xlm-v-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
labels = [
    "personal experience",
    "no personal experience",
]


In [10]:
def classify_personal_experience(text):
    if not isinstance(text, str) or text.strip() == "":
        return {"label": 0, "scores": {}}

    result = post_classifier(text, labels)

    top_label = result["labels"][0]
    score_dict = dict(zip(result["labels"], result["scores"]))
    binary_label = 1 if top_label == "personal experience" else 0

    return {"label": binary_label, "scores": score_dict}

In [11]:
texts = data_comments["raw"].fillna("")

results = []
for t in tqdm(texts, desc="Classifying posts"):
    res = classify_personal_experience(t)
    results.append(res)

data_comments["class_pers_exp"] = results
data_comments["label_pers_exp"] = data_comments["class_pers_exp"].apply(lambda x: x["label"])
data_comments["scores_pers_exp"] = data_comments["class_pers_exp"].apply(lambda x: x["scores"])

Classifying posts: 100%|██████████| 150/150 [00:14<00:00, 10.28it/s]


In [ ]:
# column with the likeliest label (string)
nyelection["stance_label"] = [r["label"] for r in results]

# column with all stance probabilities (dict)
nyelection["stance_scores"] = [r["scores"] for r in results]
